In [ ]:
import torch
import torch.nn as nn
import os
import numpy as np
import json
import math
from tqdm import tqdm
import torch.nn.functional as F

In [ ]:
weightPath = r"" # weights path folder location
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def ReadJson(filePath):
    with open(filePath, "r") as f:
        data = json.load(f)
    return data

def ReadTxt(filePath):
    data = dict()
    index = 0

    with open(filePath, "r") as f:
        for line in f:
            line = line.strip()
            data[line] = index
            index += 1
    return data

class Tokenizer:
    def __init__(self):
        token = ReadJson(r"/home/shivam/ssd_data/project/Lamma/tokenizer.json") # Add your tokenizer.json file location, you can download easily for llama 3
        self.merge = dict()
        index = 0
        for word in token["model"]["merges"]:
            self.merge[word] = index
            index += 1
        self.vocab = token["model"]["vocab"]
        self.inerseVocab = list(self.vocab.keys())

    def Encode(self, text):
        space = "Ġ"
        newLine = "Ċ"
        # text = " " + text
        text = text.replace(" ", space)
        text = text.replace("\n", newLine)
        text = list(text)
        while True:
            mergeIndex = self.FindMerge(text)
            if(mergeIndex == -1):
                break
            text[mergeIndex] += text[mergeIndex + 1]
            text.pop(mergeIndex + 1)
        output = [128000]
        for word in text:
            output.append(self.vocab[word])
        return output
    
    def Decode(self, list):
        txt = ""
        for i in list:
            txt += self.inerseVocab[i]
        return txt.replace("Ġ", " ").replace("Ċ", "\n")

    def FindMerge(self, text):
        crnIndex = len(self.merge)
        index = -1
        for i in range(len(text) - 1):
            merge = text[i] + " " + text[i+1]
            if(merge in self.merge and self.merge[merge] < crnIndex):
                crnIndex = self.merge[merge]
                index = i
        return index

In [ ]:
ropeTheta = 500000.0
qkvDim = 128
theta = torch.tensor([ropeTheta**(-2 * (i)/qkvDim) for i in range(qkvDim//2)])

def RopeMatrixPrior(seqLen):
    # Create cos and sin for whole sequence
    seqTheta = torch.stack([i * theta for i in range(seqLen)]).to(device=device)
    return torch.sin(seqTheta), torch.cos(seqTheta)

def RopeMatrix(seqLen):
    # Optimized version of above function
    seqIdx = torch.arange(seqLen, device=device).float()
    freqs = torch.outer(seqIdx, theta.to(device))
    return torch.sin(freqs), torch.cos(freqs)

In [ ]:
class Attention(nn.Module):
    def __init__(self, index):
        super().__init__()
        self.query = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.self_attn.q_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        self.key = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.self_attn.k_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        self.value = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.self_attn.v_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        self.output = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.self_attn.o_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        self.softmax = nn.Softmax(dim = -1)

    def forward(self, x, mask = None):
        # batch size one assume
        # x = (s X n)
        # mask = (s X s)
        # s = Sequence Size | n = Input Dim (for llama 4096)
        q, k, v = (self.query @ x.T).T, (self.key @ x.T).T, (self.value @ x.T).T
        seqLen, _ = x.shape
        
        # q = seqLen X numberOfHeads X queryDim
        q = q.view((seqLen, 24, 128))
        # k/v = seqLen X numberOfGroup X queryDim
        v = v.view((seqLen, 8, 128))
        k = k.view((seqLen, 8, 128))

        sinMat, cosMat = RopeMatrix(seqLen) # ROPE Vector
        # Query ROPE Implementation
        qSin = sinMat.unsqueeze(dim=1).repeat(1, 24, 1)
        qCos = cosMat.unsqueeze(dim=1).repeat(1, 24, 1)
        qStart = q[:,:,:q.shape[-1]//2] * qCos - q[:,:,q.shape[-1]//2:] * qSin
        qEnd = q[:,:,:q.shape[-1]//2] * qSin + q[:,:,q.shape[-1]//2:] * qCos
        q = torch.cat((qStart, qEnd), dim=-1)

        # Key ROPE Implementation
        kSin = sinMat.unsqueeze(dim=1).repeat(1, 8, 1)
        kCos = cosMat.unsqueeze(dim=1).repeat(1, 8, 1)
        kStart = k[:,:,:k.shape[-1]//2] * kCos - k[:,:,k.shape[-1]//2:] * kSin
        kEnd = k[:,:,:k.shape[-1]//2] * kSin + k[:,:,k.shape[-1]//2:] * kCos
        k = torch.cat((kStart, kEnd), dim=-1)

        # GQA-8 (Group Query Attention with number of groups are 8)

        k = k[:, :, None, :].expand(seqLen, 8, 3, 128).reshape(seqLen, 24, 128)
        v = v[:, :, None, :].expand(seqLen, 8, 3, 128).reshape(seqLen, 24, 128)
        
        # k = k.unsqueeze(2).repeat(1, 1, 4, 1).contiguous().view((seqLen, 32, 128)).permute(1, 0, 2)
        # v = v.unsqueeze(2).repeat(1, 1, 4, 1).contiguous().view((seqLen, 32, 128)).permute(1, 0, 2)
        q = q.permute(1, 0, 2)
        k = k.permute(1, 0, 2)
        v = v.permute(1, 0, 2)
   
        score = (q @ k.permute(0, 2, 1))/math.sqrt(128)
        if(mask is not None):
            score = score.masked_fill(mask == 0, float("-inf"))
        sf = self.softmax(score)
        
        sf = (sf @ v).permute(1,0,2).contiguous().view(seqLen, 3072)
        return sf @ self.output.T

class MLP(nn.Module):
    def __init__(self, index):
        super().__init__()
        
        # ([4096, 14336])
        self.down = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.mlp.down_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        # ([14336, 4096])
        self.up = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.mlp.up_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        # ([14336, 4096])
        self.gate = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.mlp.gate_proj.weight.npy'))).to(dtype=torch.float32).to(device=device))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x = m X n
        # m = Sequence Size | n = Token Dim (4096)
        gate = self.gate @ x.T
        up = self.up @ x.T

        gate = gate * self.sigmoid(gate) # SiLU function

        gate = gate * up
        return (self.down @ gate).T

class Transformer(nn.Module):
    def __init__(self, index):
        super().__init__()
        
        self.inpuRmsNorm = nn.RMSNorm(normalized_shape=3072, eps=1e-5)
        self.inpuRmsNorm.weight.data = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.input_layernorm.weight.npy'))).to(dtype=torch.float32).to(device=device))
        
        self.postAttentionRmsNorm = nn.RMSNorm(normalized_shape=3072, eps = 1e-5)
        self.postAttentionRmsNorm.weight.data = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, f'model.layers.{index}.post_attention_layernorm.weight.npy'))).to(dtype=torch.float32).to(device=device))
        
        self.attn = Attention(index)
        self.fnn = MLP(index)

    def forward(self, x, mask = None):
        # batch size one assume
        # x = (s X n)
        # mask = (s X s)
        attnX = self.inpuRmsNorm(x)
        attnX = self.attn(attnX, mask)

        x = x + attnX

        attnX = self.postAttentionRmsNorm(x) # using same variable name but it is for FNN
        attnX = self.fnn(attnX)

        return x + attnX

class Llama3(nn.Module):
    def __init__(self):
        super().__init__()
        self.embd = nn.Embedding(num_embeddings=50257, embedding_dim=768).to(device) # Start with random embedding size and dim
        self.embd.weight.data = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, 'model.embed_tokens.weight.npy'))).to(dtype=torch.float32).to(device=device))
        
        # self.decode = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, 'lm_head.weight.npy'))).to(dtype=torch.float32).to(device=device))

        self.layers = nn.ModuleList([
            Transformer(index) for index in range(28)
        ]) # Number of Layer

        self.rmsNorm = nn.RMSNorm(normalized_shape=4096, eps = 1e-5)
        self.rmsNorm.weight.data = nn.Parameter(torch.from_numpy(np.load(os.path.join(weightPath, 'model.norm.weight.npy'))).to(dtype=torch.float32).to(device=device))

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x : torch.tensor, mask = None):
        x = self.embd(x).to(device = device)
        for layer in self.layers:
            x = layer(x, mask).to(device=device)
        x = (self.embd.weight @ x.T).T
        return self.softmax(x)  

In [ ]:
def CreateMask(encd:list)->torch.tensor:
    n = len(encd)
    mask = torch.ones((n,n), device = device)
    mask = mask.tril()
    return mask.unsqueeze(0)

def SelectNextWord(logits, temperature=1.0, top_p=0.9):
    if logits.dim() > 1:
        logits = logits[-1]
    if top_p < 1.0:
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        indices_to_remove = sorted_indices_to_remove.scatter(0, sorted_indices, sorted_indices_to_remove)
        probs[indices_to_remove] = 0.0
        probs = probs / probs.sum()
    next_token = torch.multinomial(probs, num_samples=1)
    
    return next_token.item()

In [ ]:
lma = Llama3().eval()

In [ ]:
tkn = Tokenizer()
question = "Explain me Backpropagation"
encdList = tkn.Encode(question)
iter = 100
ans = ""
for _ in tqdm(range(iter)):
    mask = CreateMask(encdList).squeeze(dim=0)
    # print(mask.shape)
    with torch.no_grad():
        out = lma(torch.tensor(encdList).to(device=device), mask)
    # index = SelectNextWord(out)
    index = torch.max(out, dim=-1).indices[-1].item()
    encdList.append(index)
print(encdList)
print(tkn.Decode(encdList[1:]))